In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [0]:
query = """
SELECT 
    fs.order_date,
    fs.product_fk,
    fs.territory_fk,
    fs.order_quantity,
    fs.unit_price,
    fs.total_due,
    dp.product_name,
    dp.product_category_name,
    dt.country_region_name,
    ds.store_name,
    ds.business_entity_id as store_id
FROM ted_dev.marts.fact_sales fs
JOIN ted_dev.marts.dim_product dp ON fs.product_fk = dp.product_pk
JOIN ted_dev.marts.dim_territory dt ON fs.territory_fk = dt.territory_pk
LEFT JOIN ted_dev.marts.dim_store ds ON fs.sales_person_fk = ds.sales_person_id
ORDER BY fs.order_date
"""

df = spark.sql(query).toPandas()
df['order_date'] = pd.to_datetime(df['order_date'])

print(f"Dados carregados: {len(df):,} registros")
print(f"Período: {df['order_date'].min()} a {df['order_date'].max()}")
print(f"Produtos únicos: {df['product_fk'].nunique()}")
print(f"Lojas distintas: {df['store_id'].nunique()}")

monthly_data = df.groupby([
    'product_fk', 
    'store_id',
    pd.Grouper(key='order_date', freq='M')
]).agg({
    'order_quantity': 'sum',
    'unit_price': 'mean',
    'total_due': 'sum',
    'product_name': 'first',
    'store_name': 'first',
    'country_region_name': 'first'
}).reset_index()

monthly_data = monthly_data.dropna(subset=['store_id'])

In [0]:
display(monthly_data)

In [0]:
baseline_forecasts = {}
confidence_intervals = {}
window = 3 # janela da média móvel (meses)
periods = 3 # períodos de previsão (meses)
z_score_95 = 1.96 # valor crítico da distribuição normal (95%)
min_demand = 0 # demanda mínima, vendas não podem ser negativas

for (product, store), group in monthly_data.groupby(['product_fk', 'store_id']):
    if len(group) >= window:
        group = group.sort_values('order_date')
        moving_avg = group['order_quantity'].rolling(window=window).mean()
        last_ma = moving_avg.iloc[-1]
        
        if not pd.isna(last_ma):
            forecast = [max(min_demand, last_ma)] * periods
            
            historical_std = group['order_quantity'].std()
            margin_error = z_score_95 * historical_std / np.sqrt(len(group))
            lower_bound = max(min_demand, last_ma - margin_error)
            upper_bound = last_ma + margin_error
            
            baseline_forecasts[(product, store)] = {
                'model_type': 'moving_average',
                'product_name': group['product_name'].iloc[0],
                'store_name': group['store_name'].iloc[0],
                'country': group['country_region_name'].iloc[0],
                'forecast_3_months': forecast,
                'historical_data': group,
                'forecast_mean': last_ma
            }
            
            confidence_intervals[(product, store)] = {
                'forecast_mean': last_ma,
                'lower_95': lower_bound,
                'upper_95': upper_bound,
                'margin_error': margin_error
            }

print(f"Baseline: {len(baseline_forecasts)}")
print(f"Previsão média mensal: {np.mean([f['forecast_mean'] for f in baseline_forecasts.values()]):.2f} unidades")
print(f"Desvio padrão das previsões: {np.std([f['forecast_mean'] for f in baseline_forecasts.values()]):.2f}")
print(f"Previsão mínima: {np.min([f['forecast_mean'] for f in baseline_forecasts.values()]):.2f} unidades")
print(f"Previsão máxima: {np.max([f['forecast_mean'] for f in baseline_forecasts.values()]):.2f} unidades")

top_products = {}
top_stores = {}

for (product, store), data in baseline_forecasts.items():
    product_name = data['product_name']
    store_name = data['store_name']
    forecast_mean = data['forecast_mean']
    
    if product_name not in top_products or forecast_mean > top_products[product_name][1]['forecast_mean']:
        top_products[product_name] = ((product, store), data)
    
    if store_name not in top_stores or forecast_mean > top_stores[store_name][1]['forecast_mean']:
        top_stores[store_name] = ((product, store), data)

top_3_products = sorted(top_products.items(), key=lambda x: x[1][1]['forecast_mean'], reverse=True)[:3]
top_3_stores = sorted(top_stores.items(), key=lambda x: x[1][1]['forecast_mean'], reverse=True)[:3]

fig, axes = plt.subplots(2, 3, figsize=(24, 12))

all_items = [(top_3_products, "Top 3 Produtos"), (top_3_stores, "Top 3 Lojas")]

for row in range(2):
    items, section_title = all_items[row]
    
    for col in range(3):
        ax = axes[row, col]
        
        if col < len(items):
            name, ((product, store), forecast_data) = items[col]
            historical = forecast_data['historical_data']
            
            dates = historical['order_date']
            quantities = historical['order_quantity']
            ma = quantities.rolling(window=3, center=False).mean()
            
            last_date = dates.iloc[-1]
            future_dates = pd.date_range(start=last_date + pd.DateOffset(months=1), periods=3, freq='M')
            forecast_values = forecast_data['forecast_3_months']
            
            ci = confidence_intervals[(product, store)]
            
            ax.plot(dates, quantities, 'o-', color='#2E86AB', 
                    label='Histórico', alpha=0.8, markersize=4, linewidth=2)
            
            ax.plot(dates, ma, '--', color='#A23B72', 
                    label='Média Móvel (3)', alpha=0.9, linewidth=2.5)
            
            ax.plot(future_dates, forecast_values, 's-', color='#F18F01', 
                    label='Previsão', markersize=7, linewidth=3, alpha=0.9)
            
            ax.fill_between(future_dates, 
                           [ci['lower_95']] * 3,
                           [ci['upper_95']] * 3,
                           alpha=0.25, color='#F18F01', label='IC 95%')
            
            ax.axvline(x=last_date, color='gray', linestyle=':', alpha=0.6, linewidth=1)
            
            product_name = forecast_data['product_name']
            store_name = forecast_data['store_name']
            country = forecast_data['country']
            
            if row == 0:
                title = f'{product_name}'
            else:
                title = f'{store_name}'
            
            ax.set_title(title, fontweight='bold', fontsize=10, pad=15)
            
            if col == 0:
                ax.legend(loc='best', framealpha=0.9, fontsize=8, 
                         bbox_to_anchor=(0.02, 0.85), frameon=True)
            
            ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=45, labelsize=8)
            ax.tick_params(axis='y', labelsize=8)
            
            forecast_mean = forecast_data['forecast_mean']
            ax.text(0.02, 0.98, f'Prev: {forecast_mean:.1f} un/mês', 
                   transform=ax.transAxes, verticalalignment='top', fontsize=8,
                   bbox=dict(boxstyle='round,pad=0.3', alpha=0.7))
        else:
            ax.set_visible(False)

plt.suptitle('Previsões - Top 3 Produtos vs Top 3 Lojas', 
             fontsize=14, fontweight='bold', y=1.00)

axes[0, 1].text(0.5, 1.12, 'Top 3 Produtos', transform=axes[0, 1].transAxes, 
                ha='center', va='bottom', fontsize=12, fontweight='bold')

axes[1, 1].text(0.5, 1.12, 'Top 3 Lojas', transform=axes[1, 1].transAxes, 
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()